# 02 — Feature Engineering

Walks through every feature group: rolling pace anchors, tyre dynamics, weather joins, telemetry dynamics, and the 2026 active aero features. Shows why each was added and what it contributes.

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import pathlib, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, str(pathlib.Path('../src')))

from pitwall.features.pace import build_pace_features, PACE_NUMERICAL, PACE_CATEGORICAL

In [ ]:
# Load silver data
files = sorted(pathlib.Path('../data/silver/laps').rglob('*.parquet'))
silver = pl.concat([pl.read_parquet(f) for f in files])
print(f'Silver: {len(silver):,} rows  {silver["session_id"].n_unique()} sessions')

In [ ]:
# Build the full gold feature store
gold = build_pace_features(silver)
print(f'Gold matrix: {gold.shape[0]:,} rows x {gold.shape[1]} columns')
print('\nNumerical features:', PACE_NUMERICAL)
print('\nCategorical features:', PACE_CATEGORICAL)

In [ ]:
# Null counts — every feature should be 0
null_report = {}
for col in PACE_NUMERICAL + PACE_CATEGORICAL:
    if col in gold.columns:
        n = gold[col].null_count()
        null_report[col] = n
        if n > 0:
            print(f'  NULLS in {col}: {n}')
if all(v == 0 for v in null_report.values()):
    print('No nulls — point-in-time joins are clean')

In [ ]:
# Feature: rolling_median_3 vs rolling_median_5
# These are the most important predictors (ablation +4.0s MAE without them)
if 'rolling_median_3' in gold.columns and 'next_clean_lap_s' in gold.columns:
    subset = gold.filter(pl.col('next_clean_lap_s').is_not_null()).sample(n=min(3000, len(gold)), seed=42)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.scatter(subset['rolling_median_3'].to_numpy(), subset['next_clean_lap_s'].to_numpy(),
                alpha=0.15, s=8, color='#00d2be')
    ax1.set_xlabel('rolling_median_3'); ax1.set_ylabel('next_clean_lap_s'); ax1.set_title('3-lap rolling median vs target')
    ax2.scatter(subset['rolling_median_5'].to_numpy(), subset['next_clean_lap_s'].to_numpy(),
                alpha=0.15, s=8, color='#f59e0b')
    ax2.set_xlabel('rolling_median_5'); ax2.set_ylabel('next_clean_lap_s'); ax2.set_title('5-lap rolling median vs target')
    plt.suptitle('Rolling medians are the strongest pace anchor', fontsize=12, color='white')
    plt.tight_layout(); plt.show()

In [ ]:
# Hard compound non-linearity: warmup phase
# tyre_warmup_phase = 1 when lap_in_stint <= 3
if 'tyre_warmup_phase' in gold.columns and 'next_clean_lap_s' in gold.columns:
    hard = gold.filter((pl.col('compound') == 'HARD') & pl.col('next_clean_lap_s').is_not_null())
    warmup = hard.filter(pl.col('tyre_warmup_phase') == 1)['next_clean_lap_s'].drop_nulls().to_numpy()
    settled = hard.filter(pl.col('tyre_warmup_phase') == 0)['next_clean_lap_s'].drop_nulls().to_numpy()
    print('Hard compound:')
    print(f'  Warmup (laps 1-3):  median = {np.median(warmup):.2f}s  n={len(warmup)}')
    print(f'  Settled (lap 4+):   median = {np.median(settled):.2f}s  n={len(settled)}')
    print(f'  Delta: +{np.median(warmup) - np.median(settled):.2f}s during warmup')
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(warmup, bins=25, alpha=0.7, color='#ff1801', label=f'Warmup (laps 1-3)  n={len(warmup)}')
    ax.hist(settled, bins=25, alpha=0.7, color='#94a3b8', label=f'Settled (lap 4+)  n={len(settled)}')
    ax.set_xlabel('Lap time (s)'); ax.legend(); ax.set_title('Hard compound: warmup phase vs settled')
    plt.tight_layout(); plt.show()

In [ ]:
# Weather feature distributions
for feat in ['track_temp_c', 'air_temp_c', 'humidity_pct']:
    if feat in gold.columns:
        vals = gold[feat].drop_nulls().to_numpy()
        print(f'{feat:20s}: mean={vals.mean():.1f}  std={vals.std():.1f}  non-zero={np.sum(vals != 0)/len(vals):.1%}')

In [ ]:
# Why weather features don't contribute (ablation delta ~ 0)
# Most weather values are imputed from circuit defaults, not real telemetry
# Real fix: lap-level synchronised weather API data
if 'track_temp_c' in gold.columns:
    print('track_temp_c zeros:', (gold['track_temp_c'] == 0).sum(), '/', len(gold))
    print('air_temp_c zeros: ', (gold['air_temp_c'] == 0).sum(), '/', len(gold))

In [ ]:
# Correlation of numeric features with target
if 'next_clean_lap_s' in gold.columns:
    target = gold['next_clean_lap_s'].to_numpy()
    print('Feature correlations with next_clean_lap_s:')
    print('-' * 45)
    correlations = []
    for feat in PACE_NUMERICAL:
        if feat in gold.columns:
            vals = gold[feat].to_numpy()
            mask = ~np.isnan(vals) & ~np.isnan(target)
            if mask.sum() > 100:
                r = np.corrcoef(vals[mask], target[mask])[0, 1]
                correlations.append((feat, r))
    for feat, r in sorted(correlations, key=lambda x: abs(x[1]), reverse=True):
        bar = '█' * int(abs(r) * 30)
        sign = '+' if r > 0 else '-'
        print(f'  {feat:25s}  r={r:+.3f}  {sign}{bar}')